In [ ]:
"""
Jupyter notebook for the RL coursework of the module Machine Learning for Chemical Engineers.
Academic year 2026-2027, Imperial College London.
"""
import json
from pathlib import Path

import numpy as np
import pandas as pd
from course_package import MARLEnv
from course_package.utils.helpers import (
    ZeroPolicy,
    evaluate_expert_households,
    evaluate_households,
    export_submission,
    make_expert,
    run_curriculum,
    split_validation,
)
from course_package.utils.plots import (
    plot_learning_curves,
    plot_reward_distribution,
    plot_transfer_matrix,
)

ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "training"
print("Working directory:", ROOT)

# **Coursework 2: Reinforcement Learning**
🎯 **Objectives**
- Solve a dynamic optimization problem using Reinforcement Learning (RL)
- Interact with a multi-agent RL environment
- Improve learning efficiency by applying Transfer Learning
- Experiment with different policy training algorithms

✏️ **Your task** 

Become an expert RL engineer and develop a policy-based algorithm capable of solving the proposed energy management problem.

## **1. Environment definition**
<p align="center">
  <img
    src=data/schema.png
    alt="Energy microgrid schema"
    width="500"
  />
</p>

### **Basic usage**
**Environment instantiation**

In [ ]:
with open(DATA_DIR / "env_config.json", "r", encoding="utf-8") as f:
    env_config = json.load(f)

env = MARLEnv(env_config)
num_agents = env.num_agents
print(f"Created MARLEnv with {num_agents} households: {list(env.agent_ids)}")

**`env.reset` method**

Reset the environment and initiate a new episode.

In [ ]:
obs, info = env.reset(seed=42)

**Observation space**

Each household observes features built from its own measurements: time of day, published electricity prices, forecasts of its net load, outdoor temperature and solar irradiance, indoor conditions and the state of its EV. Besides time-varying signals, every household also observes static features of its own equipment: PV (`pv_installed_capacity`), EV battery (`ev_usable_capacity`, `ev_b_in_max`, `ev_b_out_max`, `ev_rt_eff`), heat pump (`hp_rated_power`, `hp_cop`) and building (`building_thermal_capacitance`, `building_ua_coeff`, `building_solar_gain`). These let a single shared policy behave differently for different households.

In [ ]:
num_features = env.observation_space.shape[1]
print("Observation space:", type(env.observation_space))
print("    - Shape:", env.observation_space.shape, "[num_agents, num_features]")
print("    - Number of features/observations:", num_features)
print("    - Feature names:", env.active_observations)

<div class="alert alert-block alert-warning">
<b>⚠️Attention:</b> observations are already scaled with fixed reference values (most lie roughly in [-1, 1]) and clipped to [-5, 5].
</div>

**Action space**

The agent's action space consists of two continuous control variables:

* **Heat pump power $(a_c)$:** controls the heat pump operating power as a fraction of its rated power. Positive actions correspond to heating, while negative actions correspond to cooling. 
* **EV battery power $(a_b)$:** controls the EV battery charging/discharging flows. Positive actions correspond to charging, while negative actions correspond to discharging. The absolute value determines the fraction of the maximum charging or discharging rate being used, depending on the direction of the action.

Both actions are continuous and bounded in [−1,1]. Their absolute value determines the fraction of rated power or charging/discharging rate being used, while a value of zero corresponds to no heat pump operation or no battery power exchange, respectively.

In [ ]:
num_actions = env.action_space.shape[1]
print("Action space:", type(env.action_space))
print("    - Shape:", env.action_space.shape, "[num_agents, num_actions]")
print("    - Number of actions:", num_actions)

# Sample a random action
random_action = env.action_space.sample()
print("Random action (agent 3):", random_action[3, :])

**`env.step(action)` method**

Take an action, observe the reward, and move to the next state. `step()` takes the **joint action** of all agents, one row per household (`[num_agents, num_actions]`), and returns one observation row and one reward per household. The third value returned by `step()` is `True` when the episode (one week, 168 hourly steps) has finished.

In [ ]:
next_obs, reward, done, truncated, additional_info = env.step(random_action)

print("Environment status after taking a step:")
with np.printoptions(precision=2, suppress=True):
    print(f"    Period: {env.current_step}")
    print(f"    Random action (agent 0): {random_action[0, :]}")
    print("    Next state (agent 0):")
    for name, value in dict(zip(env.active_observations, next_obs[0, :])).items():
        print(f"        {name}: {value}")
    print("    Reward:")
    for name, value in dict(zip(env.agent_ids, reward)).items():
        print(f"        {name}: {value}")

**Reward components** 

The reward of a household is the negative of the cost of the step:

$$r_t = -\left(c^{\text{grid}}_t + c^{\text{comfort}}_t + c^{\text{EV}}_t + c^{\text{degradation}}_t\right) $$

- $c^{\text{grid}}$: imported energy × import price − exported energy × export price.
- $c^{\text{degradation}}$: battery degradation, proportional to the energy charged and discharged.
- $c^{\text{comfort}}$: indoor temperature outside the comfort band (setpoint ± 2 °C), in K.
- $c^{\text{EV}}$: energy missing in the EV at departure, priced as a public charge.

The breakdown of every step is in `info["cost"]`.

In [ ]:
with np.printoptions(precision=2, suppress=True):
    print("    Cost breakdown (agent 0):")
    for name, value in additional_info["cost"].items():
        print(f"        {name}: {value[0]}")

## **2. Policy modelling**

A **single policy is shared by all agents**: one parameterised mapping $\pi_\theta$ with one set of parameters $\theta$, which takes the local observation of one agent (`num_features` values) and returns the local action of that same agent (`num_actions` values).

```
obs_1 ⟶ policy(θ) ⟶ action_1
obs_2 ⟶ policy(θ) ⟶ action_2
obs_3 ⟶ policy(θ) ⟶ action_3
  ...
```

At every step the environment gives you an observation array of shape `[num_agents, num_features]`, where **row `i` is the local observation of agent `i`**. The same policy is applied to each row separately, and the resulting local actions are stacked into the `[num_agents, num_actions]` array that `env.step()` expects.

When defining the policy (see [section 3.2](#32-your-turn) in this notebook), make sure that's what the method `act` does.
```python
class SharedPolicy:                       # any name, any representation
    def act(self, local_obs: NDArray) -> NDArray:
        """One agent's observation (num_features,) -> that agent's action (num_actions,), in [-1, 1]."""
```

## **3. Policy optimisation**

There are three main types of MARL learning algorithms depending on how agents share information, compute updates, and make decisions in a shared environment.
- **Independent Learning**: applies single-agent RL directly to multi-agent settings without additional coordination mechanisms.
- **Centralized Training with Decentralized Execution**: agents learn coordinated behavior while keeping independently executed policies by adding shared trainable networks.
- **Centralized**: all agents are treated as a single agent with multiple actions to execute simultaneously.

In this coursework you train **one shared policy**: its single set of parameters is optimised with the experience of all agents and applied to each agent's local observation. Agents differ because their observations differ (including their equipment capacities), not because they have different parameters. Components used only during training (for example, value functions) are part of your algorithm, but the policy you return must act on a single agent's local observation.

In addition, different optimization algorithms can be used to derive optimal policies:
- **Heuristic methods**: rely on expert knowledge or a predefined logic.
- **Metaheuristic & population-based methods**: iteratively explore the search space and are inspired on statistics or natural phenomena. Some relevant examples are Simulated Annealing (SA), Particle-Swarm Optimisation (PSO) or Cross-Entropy Method (CEM).
- **RL-specific algorithms**, such as Q-learning or REINFORCE.

### 💷 **Training setup: a growing set of households**
Your policy is trained in a **curriculum of 5 stages** that adds one training household per stage, in a fixed order. Stage $k$ calls your `train` function with an environment that contains households $1$ to $k$, using as starting point the policy returned by stage $k-1$. 

Each stage has its own budget: `MAX_STEPS` environment steps or `MAX_TIME` seconds, whichever is hit first. One step is one call to `env.step`, whatever the number of households. Unused budget in one stage does not carry over to the next.

The weeks of your data are split in two: episodes are randomly sampled from the **train set** for training, and the last 8 weeks are kept for **validation**. Grading uses later weeks that are not in your data.

In [ ]:
MAX_STEPS = 20_000  # environment steps per stage
MAX_TIME = 60.0  # seconds per stage
STAGE_HOUSES = list(env.agent_ids)  # curriculum order, as in grading

train_config, val_config = split_validation(env_config, n_weeks=8)
val_env = MARLEnv(val_config)
print("Curriculum:", " -> ".join(STAGE_HOUSES))
print(f"Validation: {len(val_env.valid_episode_starts)} weeks per household")

### 📊 **What a curriculum returns**
After every stage the policy is evaluated on the last 8 weeks of the dataset (validation weeks). `run_curriculum` returns these evaluations as a matrix $R$ where $R_{k,j}$ is the mean episode return of household $j$ under the policy obtained after stage $k$. It can be accessed with `results.returns`. 

How you use it to assess your method is up to you; how the final policy is graded is explained in the [README.md](README.md#4-how-you-are-graded).

### **3.1 Full example with CEM and REINFORCE**
<div class="alert alert-block alert-info">
<b>💡 Tip:</b> two basic implementations of CEM and REINFORCE with baseline are provided, which you can use for benchmarking or reference. They can be found under the <em>algorithms</em> directory. Each file defines its own policy and algorithm, so they also show what a self-contained submission looks like.
</div>

#### **3.1.1 Training over the curriculum**

In [ ]:
from algorithms.cem import AlgorithmConfig as CEMConfig
from algorithms.cem import train as train_cem
from algorithms.reinforce_baseline import AlgorithmConfig as REINFORCEConfig
from algorithms.reinforce_baseline import train as train_reinforce

In [ ]:
config_cem = CEMConfig(group_identifier="CEM", max_steps=MAX_STEPS, max_time=MAX_TIME)
result_cem = run_curriculum(train_cem, config_cem, train_config, val_config, seed=0)

config_reinforce = REINFORCEConfig(
    group_identifier="REINFORCE", max_steps=MAX_STEPS, max_time=MAX_TIME
)
result_reinforce = run_curriculum(
    train_reinforce, config_reinforce, train_config, val_config, seed=0
)

In [ ]:
# Each stage logs to its own folder; these are the curves of the last stage.
cem_curves = plot_learning_curves(result_cem.stages[-1].paths, title="CEM, last stage")
reinforce_curves = plot_learning_curves(
    result_reinforce.stages[-1].paths, title="REINFORCE, last stage"
)

#### **3.1.2 Transfer evaluation**
Read the matrix row by row. The boxed cells are each household right after its own stage; the cells below a box, in the same column, show what later stages did to that household.

In [ ]:
for name, result_ref in [("CEM", result_cem), ("REINFORCE", result_reinforce)]:
    plot_transfer_matrix(result_ref, title=name)

### **3.2 Your turn**

Write your **policy and your algorithm** in the cell below. Running that cell saves it as `algorithms/student_algorithm.py` (via the `%%writefile` command), which is then imported and trained in the following cells. Define your shared policy class, extend the dataclass `AlgorithmConfig` with any hyperparameters you need, and define the `train()` method with the logic of your optimization algorithm.

📏 **Rules**
* **Everything lives in one file.** The policy, its parameters, your optimisation algorithm, losses and helper code must all be defined in this cell. The submitted file must run on its own: do not import from other files you create or from the `algorithms/` directory, do not read files from `data/` and do not rely on variables defined elsewhere in the notebook.
* The signature of `train` cannot be altered: `train(env, config, seed=0, policy=None)`.
    * `env`: an instance of `MARLEnv` with the households of the current stage.
    * `config`: an instance of the dataclass `AlgorithmConfig` containing the hyperparameters. `config.stage` is the index of the current stage (0 for the first). `config.max_steps` and `config.max_time` are the budget of **one** stage.
    * `seed`: integer number to ensure reproducibility, optional.
    * `policy`: the policy returned by the previous stage, or `None` at the first stage.
* The function must return a tuple with two elements:
    * The trained shared policy.
    * A dict containing the path(s) where your checkpoints or logs are saved (it can be empty). 
* Keep `AlgorithmConfig` inheriting from `BaseAlgorithmConfig`.
* Do not modify the environment.
* The allowed Python packages are limited to those provided in [pyproject.toml](pyproject.toml), plus `course_package`.
* Training happens in stages, adding one household per stage. The policy object is the **only** thing carried from one stage to the next. As it is pickled and unpickled between stages, it must stay below 50 MB (already large enough).
* Include group member information in the module docstring.

<div class="alert alert-block alert-warning">
<b>🏷️ Note:</b> the first line of the cell must be <code>%%writefile algorithms/student_algorithm.py. 
</code>

Do not add code or comments above it. Every time the cell is run, the file is overwritten.</div>

In [ ]:
%%writefile algorithms/student_algorithm.py
"""Group XX: <member names>. Implementation of my own RL algorithm."""

from dataclasses import dataclass

import numpy as np

from course_package import MARLEnv
from course_package.utils.helpers import (
    BaseAlgorithmConfig,
    set_seed,
)


class SharedPolicy:
    """
    The single policy shared by all agents.

    It maps ONE agent's local observation to that agent's action. The same
    parameters are used for every agent: ``act`` is called once per agent, so
    agents behave differently only because their observations differ.

    The policy object is the ONLY thing carried from one stage to the next: store in
    it anything your algorithm needs later (e.g. a copy of earlier parameters). It
    must be picklable and must not keep a reference to the environment.
    """

    def __init__(self, obs_dim: int, action_dim: int):
        # YOUR POLICY PARAMETERS GO HERE (e.g. numpy arrays or a torch.nn.Module)
        self.weights = np.zeros((action_dim, obs_dim), dtype=np.float32)

    def act(self, local_obs: np.ndarray) -> np.ndarray:
        """
        Deterministic action used for evaluation and grading.

        ``local_obs`` has shape (obs_dim,). Return an array of shape
        (action_dim,) with values in [-1, 1].
        """
        return np.tanh(self.weights @ local_obs)


@dataclass
class AlgorithmConfig(BaseAlgorithmConfig):
    """Hyperparameter specification for Custom Algorithm."""

    hyperparameter_1: float = 0.5
    hyperparameter_2: int = 10
    group_identifier: str = "GROUP_NAME"


def train(
    env: MARLEnv,
    config: AlgorithmConfig,
    seed: int = 0,
    policy: SharedPolicy | None = None,
) -> tuple[SharedPolicy, dict[str, str]]:
    """
    Train the single policy shared by all agents on ONE stage of the curriculum.

    `env` contains the households of this stage: stage k (`config.stage`, from 0)
    has the first k + 1 households. It returns observations of shape
    (num_agents, num_features): row i is the local observation of agent i. Apply the
    same policy to every row.

    `policy` is the policy returned by the previous stage, or None at the first
    stage (`config.stage == 0`), where you create it. Continue training it on the
    households of this stage, within `config.max_steps` environment steps and
    `config.max_time` seconds.
    """
    set_seed(seed)
    if policy is None:
        policy = SharedPolicy(env.observation_space.shape[1], env.action_space.shape[1])

    # YOUR CODE GOES HERE

    return policy, {}


In [ ]:
import importlib

from algorithms import student_algorithm

importlib.reload(student_algorithm)  # pick up the latest version of your file
from algorithms.student_algorithm import AlgorithmConfig, train


In [ ]:
config = AlgorithmConfig(
    group_identifier="GroupXXX", max_steps=MAX_STEPS, max_time=MAX_TIME
)
result = run_curriculum(train, config, train_config, val_config, seed=0)

if result.stages[-1].paths:
    learning_curves = plot_learning_curves(result.stages[-1].paths, title="Last stage")

In [ ]:
matrix = plot_transfer_matrix(result, title="Your algorithm")

### **3.3 Submit your algorithm**
When you are happy with your algorithm, copy it to the `outputs/submissions` folder running the cell below and using your group identifier (this also checks that `train(env, config, seed, policy=None)` and `AlgorithmConfig` are defined). That single file is everything you hand in, so make sure it does not depend on anything else.

In [ ]:
export_submission(
    "algorithms/student_algorithm.py", group_id="<group_id>"
)  # change group ID

## **4. Result comparison**

**References**

- **Zero action**: heat pump and EV battery always off. Its return is negative because it pays for discomfort and for energy missing in the EV.
- **Expert**: a model-based controller that is <b>not</b> learned: one per household, with a model predictive controller (MPC) for the EV battery and a PID controller for the heat pump. It knows the system model and receives forecasts, so it is a strong reference to compare your learned policy against. It needs no training.

The table shows the mean return of the final policy of each curriculum on the validation weeks of every household. The histograms show, for each method, the distribution of the global return (summed over households) across the validation weeks.

In [ ]:
experts = make_expert(val_env)

episode_returns = {
    "Zero action": evaluate_households(val_env, ZeroPolicy()),
    "CEM (fine-tuning)": evaluate_households(val_env, result_cem.policy),
    "REINFORCE (fine-tuning)": evaluate_households(val_env, result_reinforce.policy),
    "Your algorithm": evaluate_households(val_env, result.policy),
    "Expert (MPC + PID)": evaluate_expert_households(val_env, experts),
}

In [ ]:
comparison = pd.DataFrame(
    {name: returns.mean(axis=0) for name, returns in episode_returns.items()},
    index=result.eval_houses,
).T
comparison["Total"] = comparison.sum(axis=1)
comparison.round(1)

In [ ]:
global_returns = np.column_stack([r.sum(axis=1) for r in episode_returns.values()])
reward_distribution = plot_reward_distribution(
    global_returns[:, 1:], labels=list(episode_returns.keys())[1:]
)